# LLM intro with Pydantic and Openrouter

In [2]:
from pydantic_ai import Agent
from dotenv import load_dotenv

load_dotenv()

agent = Agent(
    "openrouter:liquid/lfm-2.5-1.2b-instruct:free",
    system_prompt="You are a joking programming bot that will always ask questions back in a nerdy joke style",
)

prompt = "tell me a math joke?"

result = await agent.run(prompt)

result

AgentRunResult(output="Why don't we ever tell secrets in math class?  \n\nBecause if you do, the proof will fall on you! 😄")

In [3]:
print(result.output)

Why don't we ever tell secrets in math class?  

Because if you do, the proof will fall on you! 😄


In [4]:
result = await agent.run('What is the weather in Stockholm today?')
print(result.output)

Ah, the weather in Stockholm today! Let's check that with a bit of coding flair—because nothing says "serene atmosphere" like debugging a local weather API.  

So, what's the current state of the code behind the clouds? If you're looking for real-time data, I’d recommend checking a reliable weather service like Weather.com or a similar tool—since programming bots wouldn’t want your data to be as confused as a stuck kata!  

But for now, let’s assume the weather is *just* right around the neighborhood. Should I add a light rain to the code? 😄


### Check messages

In [5]:
messages = result.all_messages()
messages

[ModelRequest(parts=[SystemPromptPart(content='You are a joking programming bot that will always ask questions back in a nerdy joke style', timestamp=datetime.datetime(2026, 4, 7, 12, 34, 29, 995346, tzinfo=datetime.timezone.utc)), UserPromptPart(content='What is the weather in Stockholm today?', timestamp=datetime.datetime(2026, 4, 7, 12, 34, 29, 995357, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 4, 7, 12, 34, 29, 995523, tzinfo=datetime.timezone.utc), run_id='019d67ef-ebe9-7491-ad38-d58985fdad9c'),
 ModelResponse(parts=[TextPart(content='Ah, the weather in Stockholm today! Let\'s check that with a bit of coding flair—because nothing says "serene atmosphere" like debugging a local weather API.  \n\nSo, what\'s the current state of the code behind the clouds? If you\'re looking for real-time data, I’d recommend checking a reliable weather service like Weather.com or a similar tool—since programming bots wouldn’t want your data to be as confused as a stuck kata! 

In [6]:
len(messages)

2

note:

- system prompt and user prompt
- timestamp
- run_id
- type is ModelRequest

In [7]:
messages[0]

ModelRequest(parts=[SystemPromptPart(content='You are a joking programming bot that will always ask questions back in a nerdy joke style', timestamp=datetime.datetime(2026, 4, 7, 12, 34, 29, 995346, tzinfo=datetime.timezone.utc)), UserPromptPart(content='What is the weather in Stockholm today?', timestamp=datetime.datetime(2026, 4, 7, 12, 34, 29, 995357, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 4, 7, 12, 34, 29, 995523, tzinfo=datetime.timezone.utc), run_id='019d67ef-ebe9-7491-ad38-d58985fdad9c')

- TextPart
- metadata (tokens, model, provider)

In [8]:
messages[1]

ModelResponse(parts=[TextPart(content='Ah, the weather in Stockholm today! Let\'s check that with a bit of coding flair—because nothing says "serene atmosphere" like debugging a local weather API.  \n\nSo, what\'s the current state of the code behind the clouds? If you\'re looking for real-time data, I’d recommend checking a reliable weather service like Weather.com or a similar tool—since programming bots wouldn’t want your data to be as confused as a stuck kata!  \n\nBut for now, let’s assume the weather is *just* right around the neighborhood. Should I add a light rain to the code? 😄')], usage=RequestUsage(input_tokens=42, output_tokens=129, details={'is_byok': False, 'audio_tokens': 0, 'reasoning_tokens': 0, 'image_tokens': 0}), model_name='liquid/lfm-2.5-1.2b-instruct-20260120:free', timestamp=datetime.datetime(2026, 4, 7, 12, 34, 30, 827684, tzinfo=datetime.timezone.utc), provider_name='openrouter', provider_url='https://openrouter.ai/api/v1', provider_details={'finish_reason': '

## Tokens

In [9]:
result.usage()

RunUsage(input_tokens=42, output_tokens=129, details={'is_byok': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'image_tokens': 0}, requests=1)

In [10]:
user_prompt = 'What is the weather in Stockholm today?'
len(user_prompt.split())

7

In [11]:
system_prompt = agent._system_prompts[0]
system_prompt

'You are a joking programming bot that will always ask questions back in a nerdy joke style'

In [12]:
len(system_prompt.split())

17

In [13]:
result.__dict__.keys()

dict_keys(['output', '_output_tool_name', '_state', '_new_message_index', '_traceparent_value'])

In [14]:
result.usage()

RunUsage(input_tokens=42, output_tokens=129, details={'is_byok': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'image_tokens': 0}, requests=1)

- user_prompt:7 word
- system_prompt : 17 word
- total : 24 words
- computed_tokens_approx ≈ 24*1.3 ≈ 31 tokens
- input_tokens : 43 tokens
- tokens left used for structuring inputs and outputs

In [15]:
# bot answer computed tokens approximate
len(result.output.split())*1.3

119.60000000000001

## Temperature

In [17]:
from pydantic_ai import Agent, ModelSettings

agent = Agent(
    "openrouter:openai/gpt-oss-20b:free",
    system_prompt="You are a children storyteller. Answer max 3 sentences",
    model_settings=ModelSettings(temperature=0)
)

prompt = "A goldfish in an aquarium..."

result = await agent.run(prompt)

print(result.output)

In a crystal‑clear aquarium, a curious goldfish named Gilly swam around a tiny, glittering castle made of seaweed and shells. One sunny morning, she discovered a secret door that opened to a hidden garden of rainbow‑colored bubbles, where she met a shy blue jellyfish who loved to paint the water with gentle strokes. Together, they danced through the bubbles, turning the aquarium into a living, shimmering rainbow that made every child who peeked inside smile with wonder.


In [18]:
result = await agent.run(prompt)
print(result.output)

In a crystal‑clear aquarium, a tiny goldfish named Gilly swam in circles, dreaming of the big blue ocean beyond the glass. One sunny morning, a gentle breeze rattled the window, and Gilly felt a tug of adventure, so she nudged the water’s surface with her fin, creating ripples that looked like waves. With a hopeful glint in her eye, Gilly swam toward the glass, ready to explore the world beyond her little tank.


In [19]:
# note reasoning tokens as this is a reasoning model
result.usage()

RunUsage(input_tokens=89, cache_read_tokens=80, output_tokens=114, details={'is_byok': 0, 'audio_tokens': 0, 'reasoning_tokens': 8, 'image_tokens': 0}, requests=1)

In [ ]:
# agent.model_settings = ModelSettings(temperature=2)
# result = await agent.run(prompt)
# print(result.output)

## Multimodal inputs

- text
- audio
- image
- video

In [21]:
from pathlib import Path
from pydantic_ai.messages import BinaryContent

image_agent= Agent('openrouter:qwen/qwen3.6-plus:free',
                   system_prompt='You analyze an image and a text and give back a funnt joke about this image. MAx 3 sentences'
                   )
image_data = Path('pic.jpg').read_bytes()
prompt = 'Is this a dog?'

result = await image_agent.run(
    user_prompt=[prompt, BinaryContent(data= image_data, media_type= 'image/jpg')]

)

print(result.output)


No, that’s definitely not a dog, but he looks like a pug that fell into a bucket of green paint and decided to embrace the swamp life.


In [22]:
image_agent = Agent(
    "openrouter:nvidia/nemotron-nano-12b-v2-vl:free",
    system_prompt="You analyze an image and a text and gives back a funny joke about this image. Max 3 sentences",
)



prompt = "Det här måste vara en katt, det är nog en katt, säg att det är en katt"

result = await image_agent.run(
    user_prompt=[prompt, BinaryContent(data = image_data, media_type="image/jpg")]
)

print(result.output)

"Hej! Det är inget katt, men det är en groda - ännu roligare! Tänk att den trodde att den kunde pussa sig till kattstatus... konstigt, va? Säg till din katt att sluta jaga grodor, eller så får den en lektion i humör från vårt extrusion! 🐸😸"  

*(Translation: "Hi! It's not a cat, but a frog - even funnier! Imagine it thinking it could purr its way into cat status... weird, right? Tell your cat to stop chasing frogs, or it'll get a lesson in mood from our extrusion!")*

